# Лабораторна робота №2. Курс гривні до USD та EUR у 2022 році
Дані НБУ зберігаються в бакеті S3 `sikst-lab2-sviatishenko-737668` (CSV, префікс `nbu/`). Ноутбук працює на інстансі EC2 і отримує доступ до S3 через IAM-роль інстансу.

In [ ]:
import io
import boto3
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
%matplotlib inline

BUCKET = 'sikst-lab2-sviatishenko-737668'
s3 = boto3.client('s3')
keys = [o['Key'] for o in s3.list_objects_v2(Bucket=BUCKET, Prefix='nbu/')['Contents']]
keys

## Читання CSV-файлів з бакету

In [ ]:
frames = []
for key in keys:
    body = s3.get_object(Bucket=BUCKET, Key=key)['Body'].read()
    frames.append(pd.read_csv(io.BytesIO(body), parse_dates=['date']))
df = pd.concat(frames)
rates = df.pivot(index='date', columns='currency', values='rate').sort_index()
rates.head()

In [ ]:
rates.describe().round(2)

## Графік курсу гривні у 2022 році

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))
ax.plot(rates.index, rates['USD'], label='USD (долар США)', color='#1f6feb', linewidth=2)
ax.plot(rates.index, rates['EUR'], label='EUR (євро)', color='#d29922', linewidth=2)
jump = pd.Timestamp('2022-07-21')
ax.axvline(jump, color='gray', linestyle='--', linewidth=1)
ax.annotate('21.07.2022: фіксований курс USD:\n29,25 → 36,57 грн', xy=(jump, 36.57),
            xytext=(pd.Timestamp('2022-09-01'), 30.5), arrowprops=dict(arrowstyle='->', color='gray'))
ax.set_title('Офіційний курс гривні до USD та EUR, 2022 рік (дані НБУ)')
ax.set_ylabel('грн за 1 одиницю валюти')
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m.%Y'))
ax.grid(alpha=0.3)
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig('uah_rates_2022.png', dpi=150)
plt.show()

## Збереження графіка на S3

In [ ]:
s3.upload_file('uah_rates_2022.png', BUCKET, 'plots/uah_rates_2022.png')
for o in s3.list_objects_v2(Bucket=BUCKET)['Contents']:
    print(f"{o['LastModified']:%Y-%m-%d %H:%M}  {o['Size']:>7}  {o['Key']}")